# Virtual Environments — Drills

These drills are different from the other modules. You cannot unit-test "open a terminal" — so instead, you will write Python code that **inspects and validates** a live virtual environment. These are real-world scripts that professionals write for deployment checks, CI/CD pipelines, and security audits.

**Before you start:** Make sure you are running this notebook inside an active virtual environment. Run the first cell to confirm.

---

## Pre-flight Check


In [ ]:
# Run this first. If it says [WARNING], stop and activate your venv before continuing.
import sys

in_venv = sys.prefix != sys.base_prefix

if in_venv:
    print("[OK] Virtual environment is active")
    print("     Python:", sys.executable)
    print("     Prefix: ", sys.prefix)
else:
    print("[WARNING] No virtual environment detected")
    print("          Steps to fix:")
    print("          1. python -m venv .venv")
    print("          2. source .venv/bin/activate  (Linux/macOS)")
    print("             .venv\\Scripts\\Activate.ps1  (Windows PS)")
    print("          3. Register as Jupyter kernel and restart the kernel")


---

## Part A — Environment Identification


**Exercise 1**

Print the following four facts about the current Python environment on separate lines:

1. The full path to the Python executable
2. The Python version (major.minor only, e.g. `3.11`)
3. The active prefix (`sys.prefix`)
4. Whether you are inside a virtual environment (`True` or `False`)


In [ ]:
# Exercise 1
import sys


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import sys

print("Executable:  ", sys.executable)
print("Version:     ", f"{sys.version_info.major}.{sys.version_info.minor}")
print("Prefix:      ", sys.prefix)
print("In venv:     ", sys.prefix != sys.base_prefix)
```

</details>

**Exercise 2**

Read the `pyvenv.cfg` file from the active virtual environment and print its contents as a Python dictionary. Each line in the file has the format `key = value`.

Expected output format:
```
{'home': '/usr/bin', 'include-system-site-packages': 'false', 'version': '3.11.0', ...}
```


In [ ]:
# Exercise 2
import sys
from pathlib import Path


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import sys
from pathlib import Path

cfg_path = Path(sys.prefix) / "pyvenv.cfg"

config = {}
if cfg_path.exists():
    for line in cfg_path.read_text().splitlines():
        line = line.strip()
        if "=" in line:
            key, _, value = line.partition("=")
            config[key.strip()] = value.strip()

print(config)
print()
for key, value in config.items():
    print(f"  {key:<40} {value}")
```

</details>

**Exercise 3**

Find the `site-packages` directory of the active environment and count how many **package directories** it contains (directories only, not files, ignoring anything starting with `_` or ending in `.dist-info`).


In [ ]:
# Exercise 3
import site
from pathlib import Path


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import site
from pathlib import Path

site_packages = site.getsitepackages()
if not site_packages:
    print("No site-packages found")
else:
    sp_path = Path(site_packages[0])
    package_dirs = [
        d for d in sp_path.iterdir()
        if d.is_dir()
        and not d.name.startswith("_")
        and not d.name.endswith(".dist-info")
        and not d.name.endswith(".data")
    ]

    print(f"site-packages: {sp_path}")
    print(f"Package directories found: {len(package_dirs)}")
    for pkg in sorted(package_dirs):
        print(f"  {pkg.name}")
```

</details>

---

## Part B — PATH and sys.path Inspection


**Exercise 4**

Split the OS `PATH` variable into a list of individual entries. Print each entry on its own line with its index (position). Mark any entry that contains `.venv` with `<-- virtual environment`.


In [ ]:
# Exercise 4
import os


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import os

entries = os.environ.get("PATH", "").split(os.pathsep)

for i, entry in enumerate(entries):
    label = "  <-- virtual environment" if ".venv" in entry else ""
    print(f"[{i:2}] {entry}{label}")
```

</details>

**Exercise 5**

Find the position of the virtual environment's `bin` (or `Scripts`) directory in `sys.path`. Print whether it appears before or after the system site-packages. If the venv entry comes first, print `[OK] venv has import priority`. If not, print a warning.


In [ ]:
# Exercise 5
import sys


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import sys

venv_idx   = None
system_idx = None

for i, path_entry in enumerate(sys.path):
    if sys.prefix in path_entry and "site-packages" in path_entry:
        venv_idx = i
    if sys.base_prefix in path_entry and "site-packages" in path_entry:
        system_idx = i

print("sys.path entries:")
for i, p in enumerate(sys.path):
    print(f"  [{i}] {p}")

print()
if venv_idx is not None and (system_idx is None or venv_idx < system_idx):
    print(f"[OK] venv site-packages at sys.path[{venv_idx}] has import priority")
elif venv_idx is None:
    print("[INFO] No venv site-packages found in sys.path")
else:
    print(f"[WARN] venv at [{venv_idx}] appears AFTER system at [{system_idx}]")
```

</details>

**Exercise 6**

Write a function `locate_module(name)` that searches `sys.path` manually and returns the path where the module would be found (the first match), or `None` if not found. Test it with `"requests"`, `"os"`, and a package that is not installed.


In [ ]:
# Exercise 6
import sys
from pathlib import Path

def locate_module(name):
    pass   # your implementation here

print(locate_module("requests"))
print(locate_module("os"))
print(locate_module("not_a_real_package"))


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import sys
from pathlib import Path

def locate_module(name):
    """Return the path of the first matching module in sys.path, or None."""
    for search_dir in sys.path:
        if not search_dir:   # empty string means current dir
            search_dir = "."
        candidate = Path(search_dir) / name
        if candidate.exists():
            return str(candidate)
    return None

for module_name in ["requests", "os", "not_a_real_package"]:
    result = locate_module(module_name)
    if result:
        print(f"[FOUND]     {module_name:<25} -> {result}")
    else:
        print(f"[NOT FOUND] {module_name}")
```

</details>

---

## Part C — Package Management


**Exercise 7**

Use `importlib.metadata` to list all installed packages in the current environment, sorted alphabetically. Print only the name and version. Count the total.


In [ ]:
# Exercise 7
import importlib.metadata


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import importlib.metadata

packages = [
    (dist.metadata["Name"], dist.version)
    for dist in importlib.metadata.distributions()
]
packages.sort(key=lambda x: x[0].lower())

for name, version in packages:
    print(f"  {name:<30} {version}")

print()
print(f"Total packages installed: {len(packages)}")
```

</details>

**Exercise 8**

Write a function `is_package_installed(name)` that returns `True` if the package is installed and `False` if not. Test it on at least three packages — include one you know is not installed.


In [ ]:
# Exercise 8
import importlib.metadata

def is_package_installed(name):
    pass   # your implementation here


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import importlib.metadata

def is_package_installed(name):
    """Return True if the named package is installed in the current environment."""
    try:
        importlib.metadata.version(name)
        return True
    except importlib.metadata.PackageNotFoundError:
        return False

to_check = ["pip", "requests", "definitely_not_installed_xyz"]
for pkg in to_check:
    status = "installed" if is_package_installed(pkg) else "not installed"
    print(f"  {pkg:<35} {status}")
```

</details>

**Exercise 9**

Generate a `requirements.txt`-style output using `importlib.metadata` — no `subprocess`, no `pip freeze`. List each installed package in `name==version` format, sorted alphabetically. Skip packages with names starting with `_`.


In [ ]:
# Exercise 9
import importlib.metadata


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import importlib.metadata

requirements_lines = []

for dist in importlib.metadata.distributions():
    name = dist.metadata["Name"]
    if name and not name.startswith("_"):
        requirements_lines.append(f"{name}=={dist.version}")

requirements_lines.sort(key=str.lower)

for line in requirements_lines:
    print(line)
```

</details>

---

## Part D — requirements.txt Operations


**Exercise 10**

Parse the requirements string below and build a dictionary `{package_name: version}`. Handle:
- Lines starting with `#` (skip them)
- Blank lines (skip them)
- Lines starting with `-r` (skip them — they are file references)
- Lines with `==` (pinned versions — parse name and version)


In [ ]:
# Exercise 10
requirements_text = """
# Core security tools
requests==2.31.0
scapy==2.5.0

# SSH automation
paramiko==3.4.0
-r base_requirements.txt
python-nmap==1.6.0
"""

# Your code here: build the dictionary
parsed = {}


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
requirements_text = """
# Core security tools
requests==2.31.0
scapy==2.5.0

# SSH automation
paramiko==3.4.0
-r base_requirements.txt
python-nmap==1.6.0
"""

parsed = {}
for line in requirements_text.splitlines():
    line = line.strip()
    if not line or line.startswith("#") or line.startswith("-"):
        continue
    if "==" in line:
        name, version = line.split("==", 1)
        parsed[name.strip()] = version.strip()

print(f"Parsed {len(parsed)} dependencies:")
for name, version in parsed.items():
    print(f"  {name:<20} {version}")
```

</details>

**Exercise 11**

Using `subprocess` and `sys.executable`, run `pip freeze` and save the output to a string. Then compare it to the `expected_packages` list below: print which packages are present and which are missing.


In [ ]:
# Exercise 11
import subprocess
import sys

expected_packages = ["pip", "requests", "scapy"]

# Run pip freeze, capture output, check each expected package


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import subprocess
import sys

expected_packages = ["pip", "requests", "scapy"]

result = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True,
    text=True
)

freeze_output = result.stdout.lower()

for pkg in expected_packages:
    if pkg.lower() + "==" in freeze_output:
        print(f"[PRESENT] {pkg}")
    else:
        print(f"[MISSING] {pkg}  -- run: pip install {pkg}")
```

</details>

---

## Part E — Putting It Together: Environment Health Report


**Exercise 12**

Write a function `is_venv_active()` that returns `True` if running inside a virtual environment and `False` otherwise. It must also print a brief explanation of how it detected the result.


In [ ]:
# Exercise 12
import sys

def is_venv_active():
    pass   # your implementation here

result = is_venv_active()
print("Return value:", result)


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import sys

def is_venv_active():
    """
    Detect whether the current Python session is inside a virtual environment.
    Method: sys.prefix is overridden by venv activation. If it differs from
    sys.base_prefix (the original system prefix), a venv is active.
    """
    active = sys.prefix != sys.base_prefix
    if active:
        print(f"sys.prefix ({sys.prefix}) != sys.base_prefix ({sys.base_prefix})")
        print("Conclusion: virtual environment is active")
    else:
        print(f"sys.prefix ({sys.prefix}) == sys.base_prefix ({sys.base_prefix})")
        print("Conclusion: running with the system Python")
    return active

result = is_venv_active()
print("Return value:", result)
```

</details>

**Exercise 13**

Write a function `check_python_version(min_major, min_minor)` that returns `True` if the running Python version meets or exceeds the minimum. Print a clear pass/fail message.


In [ ]:
# Exercise 13
import sys

def check_python_version(min_major, min_minor):
    pass   # your implementation here

check_python_version(3, 10)   # should pass
check_python_version(3, 99)   # should fail


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import sys

def check_python_version(min_major, min_minor):
    """Return True if the running Python meets the minimum version requirement."""
    current_major = sys.version_info.major
    current_minor = sys.version_info.minor
    meets = (current_major, current_minor) >= (min_major, min_minor)
    status = "PASS" if meets else "FAIL"
    print(f"[{status}] Python {current_major}.{current_minor} "
          f"(required >= {min_major}.{min_minor})")
    return meets

check_python_version(3, 10)   # pass
check_python_version(3, 99)   # fail
```

</details>

**Exercise 14**

Using all the functions you have written in this notebook, build a complete `environment_health_report()` function that prints:

1. Whether a venv is active
2. Python version check (minimum 3.10)
3. Number of installed packages
4. Whether `pip` itself is installed
5. A final `[PASS]` or `[FAIL]` line based on all checks


In [ ]:
# Exercise 14
import sys
import site
import importlib.metadata
from pathlib import Path

def environment_health_report():
    pass   # your implementation here

environment_health_report()


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import sys
import site
import importlib.metadata
from pathlib import Path

def environment_health_report():
    """Print a full environment health report and return True if all checks pass."""
    checks = []

    print("=" * 50)
    print("ENVIRONMENT HEALTH REPORT")
    print("=" * 50)

    # Check 1: virtual environment
    in_venv = sys.prefix != sys.base_prefix
    status = "[PASS]" if in_venv else "[FAIL]"
    print(f"{status} Virtual environment active: {in_venv}")
    print(f"       Python: {sys.executable}")
    checks.append(in_venv)

    # Check 2: Python version >= 3.10
    version_ok = sys.version_info >= (3, 10)
    status = "[PASS]" if version_ok else "[FAIL]"
    print(f"{status} Python {sys.version_info.major}.{sys.version_info.minor} (>= 3.10 required)")
    checks.append(version_ok)

    # Check 3: count installed packages
    packages = list(importlib.metadata.distributions())
    print(f"[INFO]  Installed packages: {len(packages)}")

    # Check 4: pip is installed
    try:
        pip_version = importlib.metadata.version("pip")
        print(f"[PASS]  pip version: {pip_version}")
        checks.append(True)
    except importlib.metadata.PackageNotFoundError:
        print("[FAIL]  pip is not installed")
        checks.append(False)

    # Final verdict
    all_ok = all(checks)
    print()
    print("=" * 50)
    print("RESULT:", "PASS -- environment is ready" if all_ok else "FAIL -- fix the issues above")
    print("=" * 50)
    return all_ok

environment_health_report()
```

</details>

**Exercise 15**

The function below is supposed to save the current environment's package list to a `requirements.txt` file. It has three bugs. Find and fix them.


In [ ]:
# Fix the three bugs in this function
import importlib.metadata

def save_requirements(output_path):
    packages = list(importlib.metadata.distributions())

    lines = []
    for dist in packages:
        name    = dist.metadata[name]      # Bug 1
        version = dist.version
        lines.append(name + version)       # Bug 2

    lines.sort()

    with open(output_path, "r") as f:      # Bug 3
        f.write("\n".join(lines) + "\n")

    print(f"Saved {len(lines)} packages to {output_path}")

# Do not call save_requirements() yet -- fix the bugs first


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
import importlib.metadata

def save_requirements(output_path):
    packages = list(importlib.metadata.distributions())

    lines = []
    for dist in packages:
        name    = dist.metadata["Name"]    # Fix 1: "Name" must be a string key, not a variable
        version = dist.version
        lines.append(f"{name}=={version}")  # Fix 2: use == separator and f-string

    lines.sort(key=str.lower)

    with open(output_path, "w") as f:      # Fix 3: "w" for write, not "r" for read
        f.write("\n".join(lines) + "\n")

    print(f"Saved {len(lines)} packages to {output_path}")

# Test it
save_requirements("/tmp/test_requirements.txt")

# Verify the file was created and show first 5 lines
from pathlib import Path
content = Path("/tmp/test_requirements.txt").read_text().splitlines()
print(f"First 5 lines of output:")
for line in content[:5]:
    print(" ", line)
```

</details>

---

## All exercises complete?

You have built a full suite of environment inspection tools. These patterns — checking `sys.prefix`, parsing `requirements.txt`, validating installed versions, running `pip` via `subprocess` — appear in real deployment scripts, CI/CD pipelines, and automated security checks.

**Skills demonstrated:**
- `sys.prefix`, `sys.base_prefix`, `sys.path`, `sys.executable`
- `site.getsitepackages()`, `pathlib.Path`
- `importlib.metadata` — package discovery and version checking
- `subprocess` — running pip from inside Python
- File I/O — reading and writing `requirements.txt`
- Debugging — identifying type errors, wrong mode strings, missing quotes
